# Car Price Regression with Missing Data

## Small-Scale Tabular Machine Learning Project

This project uses a car-sales dataset to practice a complete tabular machine-learning workflow: inspecting missing values, preparing categorical and numerical features, encoding the data, training a regression model, and evaluating the result.

The project comes from my original Scikit-Learn coursework and has been cleaned up for portfolio presentation while preserving the original modeling approach.

The dataset contains **1,000 rows** and five columns:

- `Make`
- `Colour`
- `Odometer (KM)`
- `Doors`
- `Price`

The goal is to predict **car price** from the remaining vehicle features.

## 1. Imports

The original coursework uses pandas and NumPy for data preparation and Scikit-Learn for preprocessing and modeling.

In [1]:
import pandas as pd
import numpy as np


## 2. Load the Dataset

This version focuses on the dataset containing missing values because it demonstrates more of the preprocessing required before a machine-learning model can be trained.

In [2]:
# Import car sales missing data
car_sales_missing = pd.read_csv('car-sales-extended-missing-data.csv')
car_sales_missing.head()


,Make,Colour,Odometer (KM),Doors,Price
0,Honda,White,35431.0,4.0,15323.0
1,BMW,Blue,192714.0,5.0,19943.0
2,Honda,White,84714.0,4.0,28343.0
3,Toyota,White,154365.0,4.0,13434.0
4,Nissan,Blue,181577.0,3.0,14043.0


## 3. Inspect Missing Values

The dataset contains missing values in every column. `Make` has **49** missing values; `Colour`, `Odometer (KM)`, `Doors`, and `Price` each have **50**.

Before training the model, rows with a missing target (`Price`) are removed. Missing predictor values are handled with Scikit-Learn imputers.

In [3]:
car_sales_missing.isna().sum() # Make has 49 missing values; the other columns each have 50


Make             49
Colour           50
Odometer (KM)    50
Doors            50
Price            50
dtype: int64

In [4]:
# Drop the rows in the price column with no labels
car_sales_missing.dropna(subset=['Price'], inplace=True)
car_sales_missing.isna().sum()


Make             47
Colour           46
Odometer (KM)    48
Doors            47
Price             0
dtype: int64

## 4. Separate Features and Target

`Price` is the value to predict, so it becomes `y`. The remaining columns become the feature matrix `X`.

In [5]:
#split int0 X & y
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']


## 5. Fill Missing Feature Values

The original coursework uses `SimpleImputer` and `ColumnTransformer`:

- missing `Make` and `Colour` values are replaced with `"missing"`
- missing `Doors` values are replaced with `4`
- missing `Odometer (KM)` values are replaced with the mean

In [6]:
# Fill missing values with Scikit-Learn
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Fill categorical calues with 'missing' & numerical values with mean
cat_imputer = SimpleImputer(strategy='constant',fill_value='missing')
door_imputer = SimpleImputer(strategy='constant', fill_value=4)
num_imputer = SimpleImputer(strategy='mean')

# Define columns
cat_features = ['Make', 'Colour']
door_features = ['Doors']
num_features = ['Odometer (KM)']

# Create an imputer (something that fills missing data)
imputer = ColumnTransformer([
    ('cat_imputer', cat_imputer, cat_features),
    ('door_imputer', door_imputer, door_features),
    ('num_imputer', num_imputer, num_features)
])

# Transform the data
filled_X = imputer.fit_transform(X)
filled_X


The transformed values are placed back into a DataFrame. The original coursework contained a small column-name typo here: `Odometer (KM))`. That typo has been corrected to `Odometer (KM)`.

In [7]:
car_sales_filled = pd.DataFrame(filled_X,
                               columns=['Make', 'Colour','Doors', 'Odometer (KM)'])


In [8]:
car_sales_filled.head()


,Make,Colour,Doors,Odometer (KM)
0,Honda,White,4.0,35431.0
1,BMW,Blue,5.0,192714.0
2,Honda,White,4.0,84714.0
3,Toyota,White,4.0,154365.0
4,Nissan,Blue,3.0,181577.0


In [9]:
car_sales_filled.isna().sum()


Make             0
Colour           0
Doors            0
Odometer (KM)    0
dtype: int64

## 6. Convert Categorical Features to Numbers

Machine-learning models require numerical input. The categorical columns are converted with `OneHotEncoder`, while the odometer values pass through unchanged.

This addresses the issue encountered earlier in the coursework when a regression model could not directly convert text values such as `"Toyota"` into numbers.

In [10]:
# Turn Categories into numbers
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', 
                                  one_hot, 
                                 categorical_features)],
                                 remainder='passthrough')
transformed_X = transformer.fit_transform(car_sales_filled)
transformed_X


## 7. Train the Random Forest Regression Model

The original project uses a fixed NumPy random seed, an 80/20 train/test split, and a `RandomForestRegressor` with `n_estimators=100`.

Those modeling choices are preserved here.

In [11]:
# Now we've got our data as numbers and filled (no missing data)
# Let's fit a model
np.random.seed(42)
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(transformed_X,
                                                    y,
                                                    test_size=0.2)
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)
model.score(X_test, y_test)


0.21990196728583944

## 8. Result

The saved run of this notebook produces an **R² of about 0.22** on the test set.

That is not a strong predictive result. The value of this project is the workflow it demonstrates: handling missing tabular data, separating features and target, imputing values, encoding categorical variables, splitting the data, fitting a regression model, and evaluating it.

The score should not be presented as evidence of a highly accurate car-price model.

## 9. Limitations

- The dataset contains only 1,000 rows.
- The final test-set R² is approximately 0.22, so predictive performance is limited.
- The available features are simple: make, colour, odometer reading, and number of doors. Important pricing factors such as model, year, trim, condition, accident history, location, and market conditions are absent.
- This notebook preserves the coursework preprocessing order. The imputer and encoder are fit on the full feature set **before** the train/test split. In a stricter machine-learning workflow, the data should be split first and preprocessing should be fit only on the training data to avoid information leakage into the test set.
- No additional models, hyperparameter tuning, or new preprocessing strategies were added for this portfolio version.

## Takeaway

This project is best viewed as a learning-focused demonstration of **tabular data preparation and regression with Scikit-Learn**, rather than a production-ready car-price predictor.